# Neural importance sampling

Reproduce Figures 3(b), 5 and 6 and the timing comparison in Section 5.

In [ ]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/hnsbi_asimov")
    repo = project / "repository"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/rafaellopesdesa/hnsbi_asimov.git", str(repo)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")],
        check=True,
    )
else:
    repo = Path.cwd()
    project = repo
sys.path.insert(0, str(repo))
from utils import setup_workspace

setup_workspace(repo, project / "workspace")

## Load $q_{\boldsymbol{\phi}}$ and $r_{s,\boldsymbol{\psi}}$

In [ ]:
import gc
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
import torch
from IPython.display import display
from utils import (
    FEATURES,
    predict_with_model,
    evaluate_ratio_packs,
    load_ratio_pack,
)
from utils_nf import load_flow, train_flow
from utils_nis import ensemble_log_prob_x, ensemble_sample_x, mixture_log_weights, mix_quadratures
from utils_plotting import plot_proposal_target, plot_nis_convergence

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)
SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRESEL_MODEL_DIR = Path("models_PRESEL")
REFERENCE_FLOW_MODEL_DIR = Path("models_flows_hybrid_reference_spline16_tail5")
RATIO_MODEL_DIR = {
    "signal": Path("models_Hybrid_SigvsRef_5M_ensemble4"),
    "background": Path("models_Hybrid_BkgvsRef_5M_ensemble4"),
}
NIS_MODEL_DIR = Path("models_flows_asimov_nis_same_sample_norm_ensemble3")
NIS_PLOT_DIR = Path("plots_asimov_nis_same_sample_norm_ensemble3")
NIS_CACHE_DIR = Path("saved_asimov_nis_same_sample_norm_ensemble3")
for directory in [NIS_MODEL_DIR, NIS_PLOT_DIR, NIS_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
RATIO_EVALUATION_BATCH_SIZE = 100000
REFERENCE_FLOW_TYPE = "quadratic_spline"
REFERENCE_SAMPLING_BATCH_SIZE = 65536
ASIMOV_MU_TRUE = 1.0
MU_DESIGN = np.linspace(0.0, 3.0, 13)
MU_SCAN = np.linspace(0.0, 3.0, 61)
PILOT_EVENTS = 2000000
NIS_ENSEMBLE_SIZE = 3
NIS_TRAIN_SEED = SEED + 3201
TUNING_SEED = SEED + 3400
STUDY_SEED = SEED + 300000
DEFENSIVE_FRACTION_CANDIDATES = (0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0)
N_TUNING_REPETITIONS = 128
ACCEPTANCE_CALIBRATION_EVENTS = 250000
BENCHMARK_EVENTS = 1000000
STUDY_SAMPLE_SIZES = np.asarray([512, 1024, 2048, 4096, 8192, 16384, 32768], dtype=int)
N_REPETITIONS = 128
SHOWCASE_SAMPLE_SIZE = 2048
NIS_MODEL_CONFIG = {
    "flow_type": "quadratic_spline",
    "n_features": N_DIM,
    "n_coupling_layers": 12,
    "hidden_features": 1024,
    "hidden_layers": 4,
    "spline_num_bins": 24,
    "spline_tail_bound": 6.0,
    "dropout_probability": 0.0,
}
NIS_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": 70,
    "learning_rate": 0.0001,
    "lr_scheduler_factor": 0.2,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1e-07,
    "weight_decay": 0.0,
    "validation_fraction": 0.2,
    "patience": 10,
    "gradient_clip": None,
}

In [ ]:
PRESEL_pack = load_ratio_pack(PRESEL_MODEL_DIR, 0)


def evaluate_PRESEL_ratio(x):
    return predict_with_model(x, **PRESEL_pack)


state = np.load(PRESEL_MODEL_DIR / "selection.npz")
PRESEL_RATIO_CUT = float(state["ratio_cut"])
LAM_SIG = float(state["nis_lambda_signal"])
LAM_BKG = float(state["nis_lambda_background"])
reference_flow = [load_flow(
    "reference", model_dir=REFERENCE_FLOW_MODEL_DIR, flow_type=REFERENCE_FLOW_TYPE, device=device
)]
ratio_models = {
    s: [load_ratio_pack(path, member) for member in range(4)] for s, path in RATIO_MODEL_DIR.items()
}


def evaluate_ratio(sample_name, values, batch_size=RATIO_EVALUATION_BATCH_SIZE):
    return evaluate_ratio_packs(ratio_models[sample_name], values, batch_size)


def sample_preselected_flow(flow_pack, n_events, batch_size=65_536):
    accepted_chunks = []
    n_kept = 0
    n_generated = 0
    n_passed = 0
    while n_kept < int(n_events):
        needed = int(n_events) - n_kept
        current_batch = max(int(batch_size), min(4 * int(batch_size), 2 * needed))
        generated = ensemble_sample_x(flow_pack, current_batch, batch_size=batch_size)
        generated_df = pd.DataFrame(generated, columns=FEATURES)
        passes = evaluate_PRESEL_ratio(generated_df) >= PRESEL_RATIO_CUT
        n_generated += len(generated)
        n_passed += int(passes.sum())
        accepted_chunks.append(generated[passes])
        n_kept += int(passes.sum())
    accepted = np.concatenate(accepted_chunks, axis=0)[: int(n_events)]
    return accepted.astype(np.float32, copy=False), n_passed / n_generated


def conditional_log_prob(flow_pack, values, acceptance, batch_size=65_536):
    return (
        np.asarray(
            ensemble_log_prob_x(flow_pack, values, batch_size=batch_size),
            dtype=np.float64,
        )
        - np.log(float(acceptance))
    )

## Evaluate $A(\mathbf{x})$ with same-sample normalization and train $g_{\boldsymbol{\eta}}$ (Algorithm 3)

In [ ]:
def normalized_ratios(raw_signal, raw_background, weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / weights.sum()
    signal_norm = float(np.sum(weights * raw_signal))
    background_norm = float(np.sum(weights * raw_background))
    return raw_signal / signal_norm, raw_background / background_norm


def fit_scan_normalization_target(raw_signal, raw_background, mu_values):
    """Estimate I_mu and both normalization coefficients on the pilot."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    ratio_normalization = np.asarray([raw_signal.mean(), raw_background.mean()])
    ratio_signal = raw_signal / ratio_normalization[0]
    ratio_background = raw_background / ratio_normalization[1]
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    )
    mu_values = np.asarray(mu_values, dtype=np.float64)
    integrals = np.empty(len(mu_values))
    coefficients = np.empty((len(mu_values), 2))
    for k, mu in enumerate(mu_values):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        h_ratio = h_asimov / h_mu
        log_h_ratio = np.log(h_ratio)
        integrals[k] = np.mean(h_asimov * log_h_ratio)
        # b_s = b_s^A - b_s^T because the component shapes do not depend on mu.
        coefficients[k, 0] = np.mean(
            ratio_signal * LAM_SIG
            * (ASIMOV_MU_TRUE * (log_h_ratio + 1.0) - mu * h_ratio)
        )
        coefficients[k, 1] = np.mean(
            ratio_background * LAM_BKG * (log_h_ratio + 1.0 - h_ratio)
        )
    return {
        "ratio_normalization": ratio_normalization,
        "mu_values": mu_values,
        "integrals": integrals,
        "coefficients": coefficients,
        "scale": ASIMOV_MU_TRUE * LAM_SIG + LAM_BKG,
    }


def scan_influence_amplitude(raw_signal, raw_background, target):
    """Evaluate A_norm using fixed pilot normalizations and coefficients."""
    ratio_signal = (
        np.asarray(raw_signal, dtype=np.float64) / target["ratio_normalization"][0]
    )
    ratio_background = (
        np.asarray(raw_background, dtype=np.float64) / target["ratio_normalization"][1]
    )
    h_asimov = (
        ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    )
    amplitude_squared = np.zeros(len(ratio_signal), dtype=np.float64)
    for mu, integral, (b_signal, b_background) in zip(
        target["mu_values"], target["integrals"], target["coefficients"]
    ):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        y_mu = h_asimov * np.log(h_asimov / h_mu)
        influence_mu = (
            y_mu - integral
            - b_signal * (ratio_signal - 1.0)
            - b_background * (ratio_background - 1.0)
        )
        amplitude_squared += (influence_mu / target["scale"]) ** 2
    return np.sqrt(amplitude_squared / len(target["mu_values"]))


torch.manual_seed(SEED + 100)
pilot_values, REFERENCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    reference_flow,
    PILOT_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
pilot_raw_signal = evaluate_ratio("signal", pilot_values)
pilot_raw_background = evaluate_ratio("background", pilot_values)
NIS_TARGET_STATE = fit_scan_normalization_target(
    pilot_raw_signal, pilot_raw_background, MU_DESIGN
)
pilot_amplitude = scan_influence_amplitude(
    pilot_raw_signal, pilot_raw_background, NIS_TARGET_STATE
)
np.savez(NIS_CACHE_DIR / "pilot_normalization_target.npz", **NIS_TARGET_STATE)

importance_training_df = pd.DataFrame(pilot_values, columns=FEATURES)
importance_flow = []
for member in range(NIS_ENSEMBLE_SIZE):
    member_seed = NIS_TRAIN_SEED + member
    torch.manual_seed(member_seed)
    importance_flow.append(train_flow(
        "asimov_importance", importance_training_df, features=FEATURES,
        model_dir=NIS_MODEL_DIR / f"member_{member}",
        model_config=NIS_MODEL_CONFIG, training_config=NIS_TRAINING_CONFIG,
        device=device, sample_weights=pilot_amplitude,
        load_if_available=True, seed=member_seed,
    ))

torch.manual_seed(NIS_TRAIN_SEED + 100)
probe, IMPORTANCE_PRESEL_ACCEPTANCE = sample_preselected_flow(
    importance_flow, ACCEPTANCE_CALIBRATION_EVENTS,
    batch_size=REFERENCE_SAMPLING_BATCH_SIZE,
)
del probe, importance_training_df, pilot_values, pilot_raw_signal, pilot_raw_background, pilot_amplitude
gc.collect()

## Figure 5: $\log(g_{\boldsymbol{\eta}}/q_{\boldsymbol{\phi}})$ and $\log A$

In [ ]:
VALIDATION_EVENTS = 200_000
torch.manual_seed(STUDY_SEED + 400)
validation_values, _ = sample_preselected_flow(
    reference_flow, VALIDATION_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
validation_raw_signal = evaluate_ratio("signal", validation_values)
validation_raw_background = evaluate_ratio("background", validation_values)
validation_amplitude = scan_influence_amplitude(
    validation_raw_signal, validation_raw_background, NIS_TARGET_STATE
)
validation_log_q = conditional_log_prob(
    reference_flow, validation_values, REFERENCE_PRESEL_ACCEPTANCE
)
validation_log_g = conditional_log_prob(
    importance_flow, validation_values, IMPORTANCE_PRESEL_ACCEPTANCE
)
log_target = np.log(validation_amplitude)
log_proposal_ratio = validation_log_g - validation_log_q
log_target_centered = log_target - np.mean(log_target)
log_proposal_centered = log_proposal_ratio - np.mean(log_proposal_ratio)
rng = np.random.default_rng(STUDY_SEED + 401)
plot_indices = rng.choice(len(validation_values), size=40000, replace=False)
fig, histogram = plot_proposal_target(log_target_centered, log_proposal_centered, plot_indices)
np.savez(NIS_CACHE_DIR / "proposal_target_closure.npz", **histogram)
fig.savefig(NIS_PLOT_DIR / "proposal_target_closure.pdf", bbox_inches="tight")
plt.show()
del validation_values, validation_raw_signal, validation_raw_background

## Figure 3(b): $\widehat t_A(\mu)$ with $M=10^6$

In [ ]:
def normalized_importance_weights(log_weights):
    return np.exp(log_weights - logsumexp(log_weights))


def asimov_scan(raw_signal, raw_background, mu_values, log_weights=None):
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)
    ratio_signal, ratio_background = normalized_ratios(raw_signal, raw_background, weights)
    h_asimov = ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
    scan = []
    for mu in np.asarray(mu_values, dtype=np.float64):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        statistic = 2.0 * ((mu - ASIMOV_MU_TRUE) * LAM_SIG + integral)
        scan.append(float(statistic))
    return np.asarray(scan)


def evaluate_hybrid_ratios(values):
    return (evaluate_ratio("signal", values), evaluate_ratio("background", values))


torch.manual_seed(SEED + 600)
benchmark_values, _ = sample_preselected_flow(
    reference_flow, BENCHMARK_EVENTS, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
)
benchmark_raw_signal, benchmark_raw_background = evaluate_hybrid_ratios(benchmark_values)
del benchmark_values
gc.collect()
BENCHMARK_SCAN = asimov_scan(benchmark_raw_signal, benchmark_raw_background, MU_SCAN)
zero_index = int(np.argmin(np.abs(MU_SCAN)))
BENCHMARK_Q_ZERO = float(BENCHMARK_SCAN[zero_index])
fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.plot(MU_SCAN, BENCHMARK_SCAN, color="black", lw=2.4)
ax.axvline(ASIMOV_MU_TRUE, color="0.5", ls="--", lw=1.2)
ax.set_xlabel("$\\mu$")
ax.set_ylabel("$t_A(\\mu)$")
ax.set_title("High-statistics hybrid-model Asimov benchmark")
fig.tight_layout()
fig.savefig(NIS_PLOT_DIR / "asimov_benchmark_scan.png", dpi=160)
plt.show()

## Evaluate $G(\epsilon)$ and draw from $g_\epsilon$ (Algorithm 3)

In [ ]:
def quadrature_pool(values):
    raw_signal, raw_background = evaluate_hybrid_ratios(values)
    log_q = conditional_log_prob(reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE)
    log_g = conditional_log_prob(importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE)
    return {"signal": raw_signal, "background": raw_background,
            "log_g_over_q": log_g - log_q}


def draw_quadrature_pools(sample_size, seed):
    torch.manual_seed(seed)
    q_values, _ = sample_preselected_flow(
        reference_flow, sample_size, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    )
    torch.manual_seed(seed + 10_000)
    g_values, _ = sample_preselected_flow(
        importance_flow, sample_size, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
    )
    uniforms = np.random.default_rng(seed + 20_000).random(sample_size)
    return quadrature_pool(q_values), quadrature_pool(g_values), uniforms


epsilon_rows = []
for repetition in range(N_TUNING_REPETITIONS):
    q_pool, g_pool, uniforms = draw_quadrature_pools(
        SHOWCASE_SAMPLE_SIZE, TUNING_SEED + repetition
    )
    for epsilon in DEFENSIVE_FRACTION_CANDIDATES:
        raw_s, raw_b, log_w = mix_quadratures(q_pool, g_pool, uniforms, epsilon)
        scan = asimov_scan(raw_s, raw_b, [0.0], log_weights=log_w)
        epsilon_rows.append({"epsilon": epsilon, "repetition": repetition,
                             "q_zero": float(scan[0])})

epsilon_results = pd.DataFrame(epsilon_rows)
epsilon_summary = epsilon_results.groupby("epsilon")["q_zero"].agg(
    q0_iqr=lambda values: np.quantile(values, 0.75) - np.quantile(values, 0.25)
).reset_index()
direct_iqr = float(epsilon_summary.loc[epsilon_summary["epsilon"] == 1.0, "q0_iqr"].iloc[0])
epsilon_summary["G"] = (direct_iqr / epsilon_summary["q0_iqr"]) ** 2
DEFENSIVE_REFERENCE_FRACTION = 0.1  # Value used in the paper.
epsilon_results.to_csv(NIS_CACHE_DIR / "epsilon_tuning_results.csv", index=False)
epsilon_summary.to_csv(NIS_CACHE_DIR / "epsilon_tuning_summary.csv", index=False)
display(epsilon_summary)



def sample_defensive_proposal(n_events, seed):
    rng = np.random.default_rng(seed)
    n_reference = int(rng.binomial(int(n_events), DEFENSIVE_REFERENCE_FRACTION))
    n_importance = int(n_events) - n_reference
    torch.manual_seed(seed + 1)
    reference_values, _ = (
        sample_preselected_flow(
            reference_flow, n_reference, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
        )
        if n_reference
        else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    )
    torch.manual_seed(seed + 2)
    importance_values, _ = (
        sample_preselected_flow(
            importance_flow, n_importance, batch_size=REFERENCE_SAMPLING_BATCH_SIZE
        )
        if n_importance
        else (np.empty((0, N_DIM), dtype=np.float32), np.nan)
    )
    values = np.concatenate([reference_values, importance_values], axis=0)
    values = values[rng.permutation(len(values))]
    log_q = conditional_log_prob(reference_flow, values, REFERENCE_PRESEL_ACCEPTANCE)
    log_g = conditional_log_prob(importance_flow, values, IMPORTANCE_PRESEL_ACCEPTANCE)
    return values, mixture_log_weights(log_g - log_q, DEFENSIVE_REFERENCE_FRACTION)

## Figure 6: IQR of $q_{0,A}$, scan error and $G$

In [ ]:
maximum_study_size = int(np.max(STUDY_SAMPLE_SIZES))
rows = []
for repetition in range(N_REPETITIONS):
    q_pool, g_pool, uniforms = draw_quadrature_pools(
        maximum_study_size, STUDY_SEED + 10_000 + repetition
    )
    direct_raw_signal, direct_raw_background = q_pool["signal"], q_pool["background"]
    importance_raw_signal, importance_raw_background, importance_log_weights = mix_quadratures(
        q_pool, g_pool, uniforms, DEFENSIVE_REFERENCE_FRACTION
    )
    for sample_size in STUDY_SAMPLE_SIZES:
        sample_size = int(sample_size)
        for method, raw_s, raw_b, log_w in [
            ("Direct reference", direct_raw_signal, direct_raw_background, None),
            ("Neural importance", importance_raw_signal, importance_raw_background, importance_log_weights),
        ]:
            scan = asimov_scan(
                raw_s[:sample_size], raw_b[:sample_size], MU_SCAN,
                log_weights=None if log_w is None else log_w[:sample_size],
            )
            rows.append({
                "method": method, "repetition": repetition, "sample_size": sample_size,
                "q_zero": float(scan[zero_index]),
                "q_zero_error": float(scan[zero_index] - BENCHMARK_Q_ZERO),
                "scan_rmse": float(np.sqrt(np.mean((scan - BENCHMARK_SCAN)**2))),
            })

study_results = pd.DataFrame(rows)
study_summary = study_results.groupby(["method", "sample_size"], sort=False).agg(
    q0_mean=("q_zero", "mean"),
    q0_std=("q_zero", "std"),
    q0_iqr=("q_zero", lambda values: np.quantile(values, 0.75) - np.quantile(values, 0.25)),
    q0_rmse=("q_zero_error", lambda values: np.sqrt(np.mean(values**2))),
    scan_rmse=("scan_rmse", lambda values: np.sqrt(np.mean(values**2))),
).reset_index()
direct_iqr = study_summary[study_summary["method"] == "Direct reference"].set_index("sample_size")["q0_iqr"]
study_summary["G"] = (study_summary["sample_size"].map(direct_iqr) / study_summary["q0_iqr"]) ** 2
study_results.to_csv(NIS_CACHE_DIR / "study_results.csv", index=False)
study_summary.to_csv(NIS_CACHE_DIR / "study_summary.csv", index=False)
display(study_summary)
np.savez(
    NIS_CACHE_DIR / "benchmark_and_scans.npz", mu_scan=MU_SCAN,
    benchmark_scan=BENCHMARK_SCAN, benchmark_q0=BENCHMARK_Q_ZERO,
)
for name, fig in plot_nis_convergence(study_summary).items():
    fig.savefig(NIS_PLOT_DIR / f"{name}.pdf")
    plt.show()

## Section 5: time $\widehat t_A(\mu)$ at matched IQR

In [ ]:
from scipy.optimize import minimize_scalar
from time import perf_counter

FIT_TIMING_REPETITIONS = 200
FIT_TIMING_WARMUPS = 5
END_TO_END_TIMING_REPETITIONS = 8
END_TO_END_TIMING_WARMUPS = 1
MINIMIZER_BOUNDS = (float(np.min(MU_SCAN)), float(np.max(MU_SCAN)))
MINIMIZER_XATOL = 1e-07
nis_precision_row = study_summary[
    (study_summary["method"] == "Neural importance")
    & (study_summary["sample_size"] == SHOWCASE_SAMPLE_SIZE)
].iloc[0]
direct_precision_candidates = study_summary[study_summary["method"] == "Direct reference"].copy()
direct_precision_candidates["match_distance"] = np.abs(
    np.log(direct_precision_candidates["q0_iqr"] / float(nis_precision_row["q0_iqr"]))
)
direct_precision_row = direct_precision_candidates.sort_values("match_distance").iloc[0]
MATCHED_NIS_SIZE = int(nis_precision_row["sample_size"])
MATCHED_DIRECT_SIZE = int(direct_precision_row["sample_size"])


def prepare_asimov_objective(raw_signal, raw_background, log_weights=None):
    """Return the un-clipped finite-quadrature Asimov objective."""
    raw_signal = np.asarray(raw_signal, dtype=np.float64)
    raw_background = np.asarray(raw_background, dtype=np.float64)
    if log_weights is None:
        weights = np.full(len(raw_signal), 1.0 / len(raw_signal))
    else:
        weights = normalized_importance_weights(log_weights)
    ratio_signal, ratio_background = normalized_ratios(raw_signal, raw_background, weights)
    h_asimov = ASIMOV_MU_TRUE * LAM_SIG * ratio_signal + LAM_BKG * ratio_background

    def objective(mu):
        h_mu = mu * LAM_SIG * ratio_signal + LAM_BKG * ratio_background
        integral = np.sum(weights * h_asimov * np.log(h_asimov / h_mu))
        return float(2.0 * ((mu - ASIMOV_MU_TRUE) * LAM_SIG + integral))

    return objective


def minimize_asimov(raw_signal, raw_background, log_weights=None):
    objective = prepare_asimov_objective(raw_signal, raw_background, log_weights=log_weights)
    return minimize_scalar(
        objective, bounds=MINIMIZER_BOUNDS, method="bounded", options={"xatol": MINIMIZER_XATOL}
    )


def synchronize_accelerator():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def time_repeated(call, repetitions, warmups):
    for i in range(warmups):
        call(i)
    durations = []
    for i in range(repetitions):
        synchronize_accelerator()
        start = perf_counter()
        call(i + warmups)
        synchronize_accelerator()
        durations.append(1000 * (perf_counter() - start))
    return np.median(durations)


def construct_and_minimize(method, n_events, seed):
    if method == "Direct reference":
        torch.manual_seed(seed)
        values, _ = sample_preselected_flow(reference_flow, n_events)
        log_weights = None
    else:
        values, log_weights = sample_defensive_proposal(n_events, seed)
    rs, rb = evaluate_hybrid_ratios(values)
    return minimize_asimov(rs, rb, log_weights)


timing_rows = []
for method, size, rs, rb, log_weights, precision in [
    (
        "Direct reference",
        MATCHED_DIRECT_SIZE,
        direct_raw_signal,
        direct_raw_background,
        None,
        direct_precision_row,
    ),
    (
        "Neural importance",
        MATCHED_NIS_SIZE,
        importance_raw_signal,
        importance_raw_background,
        importance_log_weights,
        nis_precision_row,
    ),
]:
    args = (rs[:size], rb[:size], None if log_weights is None else log_weights[:size])
    scan_ms = time_repeated(
        lambda i: asimov_scan(args[0], args[1], MU_SCAN, args[2]),
        FIT_TIMING_REPETITIONS,
        FIT_TIMING_WARMUPS,
    )
    minimum_ms = time_repeated(
        lambda i: minimize_asimov(*args), FIT_TIMING_REPETITIONS, FIT_TIMING_WARMUPS
    )
    full_ms = time_repeated(
        lambda i: construct_and_minimize(method, size, STUDY_SEED + 40000 + i),
        END_TO_END_TIMING_REPETITIONS,
        END_TO_END_TIMING_WARMUPS,
    )
    timing_rows.append(
        {
            "method": method,
            "M": size,
            "q0_iqr": precision["q0_iqr"],
            "scan_rmse": precision["scan_rmse"],
            "scan_ms": scan_ms,
            "minimum_ms": minimum_ms,
            "construction_minimum_ms": full_ms,
        }
    )
timing_results = pd.DataFrame(timing_rows)
for column in ["scan_ms", "minimum_ms", "construction_minimum_ms"]:
    timing_results[column.replace("_ms", "_speedup")] = (
        timing_results.loc[0, column] / timing_results[column]
    )
display(timing_results)
timing_results.to_csv(NIS_CACHE_DIR / "matched_precision_timing.csv", index=False)